# Pilot Test: Jupyter + Public LLM Memory Dump Feasibility

이 노트북은 사용자가 지정한 memory dump를 읽고, 로컬 분석 결과를 공개 LLM API에 연결하는 최소 동작 데모입니다.

In [1]:
from pathlib import Path
import json
import os
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DUMP_PATH = os.environ.get('DUMP_PATH', '').strip()
VMLINUX_PATH = os.environ.get('VMLINUX_PATH', '').strip()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DUMP_PATH =', DUMP_PATH)
print('VMLINUX_PATH =', VMLINUX_PATH or '(not provided)')
print('dump exists =', Path(DUMP_PATH).exists() if DUMP_PATH else False)
print('OPENAI_API_KEY configured =', bool(os.environ.get('OPENAI_API_KEY')))

PROJECT_ROOT = /home/taejin/Jupyter/jupyter-ramdump-analyzer
DUMP_PATH = /home/taejin/Jupyter/data/memory/memory.vmem
VMLINUX_PATH = (not provided)
dump exists = True
OPENAI_API_KEY configured = True


## Imports and runtime options

In [2]:
from analysis_pipeline import FeasibilityOptions, run_feasibility_analysis
from llm_assistant import LLMAssistant
from memory_kernel_analyzer import build_analysis_context, summarize_findings

MODEL = os.environ.get('OPENAI_MODEL', 'openai/gpt-oss-120b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')
ALLOW_RAW_CHUNK_EXCERPT = True
GENERATE_NEXT_STEPS = False

assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('MODEL =', MODEL)
print('API_BASE =', API_BASE)
assistant.is_configured()

MODEL = openai/gpt-oss-120b:free
API_BASE = https://openrouter.ai/api/v1


True

## Step 1. Build local analysis context

In [3]:
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 2. Inspect chunk samples sent to the LLM

In [4]:
for sample in context.raw_chunk_samples[:3]:
    print(sample)
    print('-' * 80)

{'offset': 0, 'size': 256, 'hex_preview': '53ff00f053ff00f0c3e200f053ff00f053ff00f054ff00f0888400f053ff00f0', 'ascii_preview': 'S...S.......S...S...T.......S...........V...V...V...V...W...........M...A.......9...Y...."J.....Y.......n...S...S.......'}
--------------------------------------------------------------------------------
{'offset': 536870912, 'size': 256, 'hex_preview': '280e3f001b0129001f0530001b0129001f0530001b0129001f0530001f053000', 'ascii_preview': '(.?...)...0...)...0...)...0...0...)...0...0...)...0...)...0...)...0...)...0...)...0...0...0...0...)...0...)...0...0...).'}
--------------------------------------------------------------------------------
{'offset': 1073741824, 'size': 256, 'hex_preview': 'fe86fe2124ed8f0525bda90b25ce981122bd0b1425fd8d1a252d872025ce8526', 'ascii_preview': '...!$...%...%..."...%...%-. %..&"(....)%../%|."..8%..>%..D%l.".x"..P%=.V%].\\%].b%..h%}.n%M.t%..z%...%.."...%...%.."...L.'}
-------------------------------------------------------------------

## Step 3. Run the feasibility pipeline

In [5]:
options = FeasibilityOptions(
    dump_path=DUMP_PATH,
    allow_raw_chunk_excerpt=ALLOW_RAW_CHUNK_EXCERPT,
    raw_chunk_limit=2,
    generate_next_steps=GENERATE_NEXT_STEPS,
)
report = run_feasibility_analysis(DUMP_PATH, assistant, options)
report.llm_enabled

True

## Step 4. Review local findings

In [6]:
print(json.dumps(report.findings, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 5. Review LLM output

API 키가 없으면 이 셀은 `None` 또는 오류 메시지를 보여줍니다.

In [7]:
print(report.llm_analysis)
print('\n' + '=' * 80 + '\n')
print(report.next_steps)

### 1️⃣ 핵심 이상 징후 (Observed Anomalies)

| 구분 | 내용 | 근거 |
|------|------|------|
| **커널 패닉 패턴** | `kernel_oops:BUG:` 와 `crashes:crash` 가 2회 발견 | `panic_pattern_count: 2` 로, OOPS/BUG 발생이 최소 2번 이상 기록됨 |
| **메모리 할당 오류 문자열** | `alloc magic is broken at %p: %lx` , `out of memory` | 메모리 관리 구조체(예: `kmalloc`, `slab`)가 손상됐을 가능성을 시사 |
| **GRUB 메모리 함수 호출 흔적** | `grub_calloc`, `grub_malloc`, `grub_realloc`, `grub_zalloc` | 부팅 단계에서 메모리 할당이 비정상적으로 진행됐을 가능성(부팅 이미지 손상 또는 하드웨어 오류) |
| **PXE‑E20 BIOS 복사 오류** | `PXE‑E20: BIOS extended memory copy error.` | 네트워크 부팅 시 메모리 복사 실패, 물리 메모리 혹은 BIOS/UEFI 펌웨어 결함을 암시 |
| **다량의 외부 IP/포트** | 17개의 외부 IP, 20‑443 포트 등 다중 네트워크 트래픽 | 시스템이 외부와 과다하게 통신하고 있거나, 침해 시도/악성 코드가 메모리 내에 존재할 가능성 |
| **프로세스/사용자 비정상** | 전체 50개의 프로세스가 단일 사용자(`woreilly`) 아래 실행 | 비정상적인 서비스/데몬이 동일 사용자 계정에 몰려 있거나, 권한 상승 후 프로세스가 재배치됐을 가능성 |
| **Hex 샘플에서 반복 패턴** | `53ff00f0…` , `1f053000…` 같은 4‑byte 반복 | 커널 구조체(예: `list_head`, `timer_list`)가 손상돼 동일 패턴이 메모리 전역에 퍼짐. 특히 `0x53ff00f0` 은 `0xF0 0x00 0xFF 0x53` 로, `0

## Step 6. Script generation example

In [8]:
if assistant.is_configured():
    script = assistant.generate_analysis_script(
        'memory dump에서 panic signal과 call trace 후보를 추출하는 Python 스크립트',
        findings=report.findings,
    )
    print(script)
else:
    print('OPENAI_API_KEY is not configured')

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
memory_dump_panic_extractor.py
- 대용량 메모리 덤프(.vmem) 파일에서 커널 패닉 시그널과
  가능한 call trace 문자열을 추출한다.
- vmlinux 심볼이 없으므로 단순 문자열/패턴 매칭에 의존한다.
- 파일을 청크 단위로 읽어 메모리 사용량을 최소화한다.
"""

import sys
import re
import argparse
from pathlib import Path

# ----------------------------------------------------------------------
# 기본 설정 (필요에 따라 커맨드라인 옵션으로 오버라이드 가능)
# ----------------------------------------------------------------------
DEFAULT_CHUNK_SIZE = 4 * 1024 * 1024   # 4 MiB
PANIC_PATTERNS = [
    b'kernel_oops:BUG:',
    b'crash',
    b'BUG:',
    b'kernel panic',
    b'Kernel panic',
    b'panic',
]
# call‑trace 라인에 흔히 보이는 prefix
CALL_TRACE_PREFIXES = [
    b'Call Trace:',
    b'---[ end trace',
    b'---[ end kernel',
    b'---[ end ',
    b'---[ ',
    b'  [<',          # backtrace entry in oops
    b'  [',
    b'    [',
]

# ----------------------------------------------------------------------
# 헬퍼 함수
# --------------------------------